# Assignment – Detecting Abnormal Patterns in Continuous Patient Monitoring (VitalDB)

**Student:** Anbuselvam K  ·  **Notebook 2 of 2** (continues `01_EDA_VitalDB.ipynb`)

**Goal.** Separate **sensor artifacts** (an isolated spike on one signal) from **real physiological events** (several vitals changing together), using the five vitals HR, SpO₂, MAP, Temperature and RR.

**Method in one paragraph.** Every reading is compared with the patient's own baseline (median of the previous 5 minutes). A reading is *flagged* when it departs from that baseline by more than K × a patient-specific robust scale. Consecutive flagged samples form an *episode*. If other vitals are flagged within ±30 s the episode is a **real event**; if it is isolated and ≤ 30 s it is an **artifact**; otherwise it goes to **review**.

**Evaluation.** VitalDB has no artifact/event labels, so known artifacts and known multi-vital events are *injected* into held-out patients and the detector is scored on finding and correctly naming them. It is compared with two baselines (clinical thresholds, Isolation Forest).

*Run all cells (Runtime → Run all). Takes a few minutes.*

In [ ]:
%pip install -q vitaldb scikit-learn

## 1. Code (same files as `src/` in the GitHub repository)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
"""Loading and cleaning of the VitalDB sample.

The cleaning rules are exactly the ones used in notebooks/01_EDA_VitalDB.ipynb:
impossible values -> NaN, temperature probe warm-up -> NaN, short gaps bridged
by carrying the last reading forward inside each patient.
"""
import numpy as np
import pandas as pd

VITALS = ["HR", "SpO2", "MAP", "Temp", "RR"]
TRACKS = {"HR": "Solar8000/HR", "SpO2": "Solar8000/PLETH_SPO2", "MAP": "Solar8000/ART_MBP",
          "Temp": "Solar8000/BT", "RR": "Solar8000/RR_CO2"}
UNITS = {"HR": "beats/min", "SpO2": "%", "MAP": "mmHg", "Temp": "°C", "RR": "breaths/min"}
INTERVAL = 2                      # seconds between samples
WARMUP_S = 600                    # temperature probe warm-up (first 10 minutes)
PLAUSIBLE = {"HR": (20, 250), "SpO2": (50, 100), "MAP": (20, 250), "Temp": (25, 45), "RR": (1, 60)}
ABNORMAL = {"HR": (50, 100), "SpO2": (92, np.inf), "MAP": (60, 120), "Temp": (35, 38), "RR": (8, 20)}
GAP_S = {"HR": 10, "SpO2": 10, "MAP": 10, "Temp": 120, "RR": 20}   # seconds a reading may be carried forward


def load_vitaldb(n_cases=30, seed=42):
    """Download n_cases random surgical cases that have all five vitals (needs internet + `pip install vitaldb`)."""
    import vitaldb

    tnames = ",".join(TRACKS.values())
    caseids = vitaldb.find_cases(tnames)
    chosen = sorted(np.random.default_rng(seed).choice(caseids, size=min(n_cases, len(caseids)), replace=False))
    frames = []
    for cid in chosen:
        arr = vitaldb.load_case(int(cid), list(TRACKS.values()), INTERVAL)
        d = pd.DataFrame(arr, columns=VITALS)
        d.insert(0, "time_s", np.arange(len(d)) * INTERVAL)
        d.insert(0, "caseid", int(cid))
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    try:
        try:
            clin = pd.read_csv("https://api.vitaldb.net/cases")
        except Exception:
            clin = vitaldb.load_clinical_data()
        keep = [c for c in ["caseid", "age", "sex", "department", "asa"] if c in clin.columns]
        df = df.merge(clin[keep], on="caseid", how="left")
    except Exception as e:                                   # clinical table is optional
        print("Clinical table not available, continuing without it:", e)
    return df


def clean(df_raw):
    """Return (df_ts, n_impossible). df_ts keeps gaps that could not be bridged (time-series friendly)."""
    df = df_raw.copy()
    n_impossible = 0
    for v, (lo, hi) in PLAUSIBLE.items():
        bad = (df[v] < lo) | (df[v] > hi)
        n_impossible += int(bad.sum())
        df.loc[bad, v] = np.nan
    df.loc[df["time_s"] < WARMUP_S, "Temp"] = np.nan
    for v in VITALS:
        limit = max(1, GAP_S[v] // INTERVAL)
        df[v] = df.groupby("caseid")[v].transform(lambda s: s.ffill(limit=limit))
    return df.reset_index(drop=True), n_impossible

In [ ]:
"""Artifact-vs-real-event detector (rule based, no training needed).

Idea
----
1. Every vital is compared with the patient's own recent baseline (median of the
   previous 5 minutes). A reading is *flagged* when it departs from that baseline by more
   than K x a robust, patient-specific scale.
2. Consecutive flagged samples of one vital form an *episode*.
3. Each episode is classified by looking at the other vitals around it:
     - other vitals also flagged within +-30 s  -> "real_event"  (several signals move together)
     - nobody else flagged and duration <= 30 s -> "artifact"    (isolated spike on one sensor)
     - otherwise                                -> "review"      (single-vital but sustained: unclear)
"""
import numpy as np
import pandas as pd


ARTIFACT, REAL, REVIEW = "artifact", "real_event", "review"
FLOOR = {"HR": 3.0, "SpO2": 1.5, "MAP": 5.0, "Temp": 0.2, "RR": 1.5}   # minimum scale per vital (units of the vital)


def compute_residuals(df, win=150, min_periods=30, vitals=VITALS):
    """Deviation from the patient's trailing baseline (median of previous `win` samples = 5 min)
    and a robust per-patient scale. df must have a RangeIndex and be sorted by caseid, time."""
    resid = pd.DataFrame(index=df.index, columns=vitals, dtype=float)
    scale = pd.DataFrame(index=df.index, columns=vitals, dtype=float)
    for v in vitals:
        base = df.groupby("caseid")[v].transform(lambda s: s.shift(1).rolling(win, min_periods=min_periods).median())
        r = df[v] - base
        mad = r.abs().groupby(df["caseid"]).transform("median").fillna(0)
        resid[v] = r
        scale[v] = np.maximum(FLOOR[v], 1.4826 * mad)
    return resid, scale


def make_flags(resid, scale, K=4.0):
    """Boolean table: True where |deviation| > K x scale."""
    return (resid.abs() > K * scale).fillna(False)


def extract_episodes(df, resid, flags, max_gap=2, co_win=15, max_artifact_s=30):
    """Group flagged samples into episodes and classify them (see module docstring)."""
    vit = list(flags.columns)
    F_all = flags.to_numpy()
    R_all = resid.to_numpy(float)
    t_all = df["time_s"].to_numpy()
    rows = []
    for cid, pos in df.groupby("caseid").indices.items():
        F, R, t = F_all[pos], R_all[pos], t_all[pos]
        n = len(F)
        cs = np.vstack([np.zeros((1, F.shape[1]), dtype=int), np.cumsum(F, axis=0)])      # prefix sums
        for j, v in enumerate(vit):
            p = np.flatnonzero(F[:, j])
            if p.size == 0:
                continue
            for g in np.split(p, np.flatnonzero(np.diff(p) > max_gap + 1) + 1):
                s, e = int(g[0]), int(g[-1])
                lo, hi = max(0, s - co_win), min(n - 1, e + co_win)
                cnt = cs[hi + 1] - cs[lo]
                others = [vit[k] for k in range(len(vit)) if k != j and cnt[k] > 0]
                dur = (e - s + 1) * INTERVAL
                if others:
                    label = REAL
                elif dur <= max_artifact_s:
                    label = ARTIFACT
                else:
                    label = REVIEW
                seg = R[s:e + 1, j]
                peak = float(seg[np.nanargmax(np.abs(seg))]) if np.isfinite(seg).any() else np.nan
                rows.append({"caseid": cid, "vital": v, "start_i": s, "end_i": e, "start_s": int(t[s]),
                             "end_s": int(t[e]), "duration_s": dur, "peak_dev": round(peak, 2),
                             "n_other": len(others), "others": ",".join(others), "label": label})
    cols = ["caseid", "vital", "start_i", "end_i", "start_s", "end_s", "duration_s", "peak_dev",
            "n_other", "others", "label"]
    return pd.DataFrame(rows, columns=cols)


def detect(df, K=4.0, **kw):
    """Convenience wrapper: returns (flags, episodes, resid)."""
    resid, scale = compute_residuals(df)
    flags = make_flags(resid, scale, K)
    return flags, extract_episodes(df, resid, flags, **kw), resid

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
"""Evaluation of the detector.

VitalDB has no labels for "artifact" or "real event", so the detector is tested with
*injected* ground truth: known artifacts (short single-vital spikes) and known physiological
events (several vitals moving together for 1-3 minutes) are added to real recordings.
The test patients are separate from the patients used to fit the Isolation-Forest baseline.

Caveat: naturally occurring anomalies in the recordings are unlabeled, so they count as false
alarms in the row-level precision of EVERY method. Use the numbers to compare methods and
settings with each other, not as absolute clinical accuracy.
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest


ART_RANGE = {"HR": (20, 70), "SpO2": (8, 25), "MAP": (25, 70), "Temp": (1.0, 3.0), "RR": (8, 20)}
EVENT_TYPES = {
    "hypotension":  {"MAP": (-40, -20), "HR": (12, 30)},
    "hypoxaemia":   {"SpO2": (-15, -7), "HR": (10, 20), "RR": (5, 10)},
    "hypertension": {"MAP": (30, 50), "HR": (15, 30)},
}


def inject(df, rng, n_art=15, n_evt=8):
    """Return (df_injected, truth[n x V int8: 0 none, 1 artifact, 2 event], list of injections)."""
    d = df.copy()
    V = len(VITALS)
    X = d[VITALS].to_numpy(float).copy()
    truth = np.zeros((len(d), V), dtype=np.int8)
    inj = []
    for cid, pos in d.groupby("caseid").indices.items():
        n = len(pos)
        if n < 1200:
            continue
        occ = np.zeros(n, bool)
        jobs = ["art"] * n_art + ["evt"] * n_evt
        rng.shuffle(jobs)
        for kind in jobs:
            for _ in range(60):
                if kind == "art":
                    j = int(rng.integers(V)); span = int(rng.integers(1, 7)); vs = [j]
                else:
                    typ = str(rng.choice(list(EVENT_TYPES))); plateau = int(rng.integers(30, 91))
                    span = plateau + 10; vs = [VITALS.index(v) for v in EVENT_TYPES[typ]]
                s = int(rng.integers(200, n - span - 200))
                lo, hi = s - 200, s + span + 200
                if occ[lo:hi].any() or np.isnan(X[pos[lo:hi]][:, vs]).any():
                    continue
                if kind == "art":
                    v = VITALS[j]; delta = rng.uniform(*ART_RANGE[v]) * (-1 if v == "SpO2" else rng.choice([-1, 1]))
                    X[pos[s:s + span], j] += delta
                    truth[pos[s:s + span], j] = 1
                    rec = {"kind": ARTIFACT, "vitals": [v], "type": "spike"}
                else:
                    shape = np.concatenate([np.linspace(0, 1, 5), np.ones(plateau), np.linspace(1, 0, 5)])
                    for v, rg in EVENT_TYPES[typ].items():
                        jj = VITALS.index(v); X[pos[s:s + span], jj] += rng.uniform(*rg) * shape
                        truth[pos[s:s + span], jj] = 2
                    rec = {"kind": REAL, "vitals": list(EVENT_TYPES[typ]), "type": typ}
                for jj in vs:                                       # stay inside physically possible values
                    v = VITALS[jj]; X[pos[s:s + span], jj] = np.clip(X[pos[s:s + span], jj], *PLAUSIBLE[v])
                occ[lo:hi] = True
                inj.append({"caseid": cid, "start_i": s, "end_i": s + span - 1, **rec})
                break
    d[VITALS] = X
    return d, truth, inj


def match_episodes(inj, ep):
    """Predicted class of every injection ('missed' if nothing overlaps) + the unmatched (background) episodes."""
    by_case = {c: g for c, g in ep.groupby("caseid")}
    used, preds = set(), []
    for I in inj:
        g = by_case.get(I["caseid"]); pred = "missed"
        if g is not None:
            m = g[g["vital"].isin(I["vitals"]) & (g["start_i"] <= I["end_i"]) & (g["end_i"] >= I["start_i"])]
            if len(m):
                used.update(m.index); labs = set(m["label"])
                pred = REAL if REAL in labs else (ARTIFACT if ARTIFACT in labs else REVIEW)
        preds.append(pred)
    return preds, ep.loc[~ep.index.isin(used)]


def prf(truth, pred):
    tp = int((truth & pred).sum()); fp = int((~truth & pred).sum()); fn = int((truth & ~pred).sum())
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(2 * p * r / max(p + r, 1e-9), 3)}


def _features(df, resid, med):
    return np.hstack([df[VITALS].fillna(med).to_numpy(float), resid.fillna(0).to_numpy(float)])


def run_evaluation(df_ts, K=4.0, seed=0, K_grid=(3, 4, 5, 6, 8)):
    rng = np.random.default_rng(seed)
    cases = np.array(sorted(df_ts.caseid.unique())); rng.shuffle(cases)
    n_test = max(2, int(round(0.3 * len(cases))))
    test_cases, train_cases = set(cases[:n_test]), set(cases[n_test:])
    out = {"n_train_cases": len(train_cases), "n_test_cases": len(test_cases), "K": K}

    dtest = df_ts[df_ts.caseid.isin(test_cases)].reset_index(drop=True)
    dtrain = df_ts[df_ts.caseid.isin(train_cases)].reset_index(drop=True)
    dtest_i, truth, inj = inject(dtest, rng)
    dtrain_i, _, _ = inject(dtrain, rng)
    hours = len(dtest_i) * 2 / 3600
    truth_row = (truth > 0).any(axis=1)
    out["n_injected_artifacts"] = sum(i["kind"] == ARTIFACT for i in inj)
    out["n_injected_events"] = sum(i["kind"] == REAL for i in inj)
    out["test_hours"] = round(hours, 1)

    resid, scale = compute_residuals(dtest_i)

    # ---- our detector at several K (sensitivity) -------------------------------------------------
    sens, main = [], None
    for k in sorted(set(K_grid) | {K}):
        flags = make_flags(resid, scale, k)
        ep = extract_episodes(dtest_i, resid, flags)
        preds, bg = match_episodes(inj, ep)
        kinds = [i["kind"] for i in inj]
        det = np.mean([p != "missed" for p in preds])
        correct = np.mean([p == kd for p, kd in zip(preds, kinds)])
        row = prf(truth_row, flags.any(axis=1).to_numpy())
        sens.append({"K": k, "detection_recall": round(det, 3), "correct_class_rate": round(correct, 3),
                     "background_episodes_per_hour": round(len(bg) / hours, 1), "row_f1": row["f1"]})
        if k == K:
            main = (flags, ep, preds, kinds, row)
    out["sensitivity"] = sens
    flags, ep, preds, kinds, row = main
    classes = [ARTIFACT, REAL, REVIEW, "missed"]
    conf = pd.DataFrame([[sum(1 for p, kd in zip(preds, kinds) if kd == t and p == c) for c in classes]
                         for t in (ARTIFACT, REAL)], index=["true_artifact", "true_real_event"],
                        columns=["pred_artifact", "pred_real_event", "pred_review", "missed"])
    out["confusion"] = conf
    out["ours_row_level"] = row
    for t, name in ((ARTIFACT, "artifact"), (REAL, "real_event")):
        idx = [i for i, kd in enumerate(kinds) if kd == t]
        out[f"{name}_detected"] = round(float(np.mean([preds[i] != "missed" for i in idx])), 3)
        out[f"{name}_classified_correctly"] = round(float(np.mean([preds[i] == t for i in idx])), 3)

    # ---- baselines (row level, same test rows) ----------------------------------------------------
    clin = np.zeros(len(dtest_i), bool)
    for v, (lo, hi) in ABNORMAL.items():
        clin |= ((dtest_i[v] < lo) | (dtest_i[v] > hi)).to_numpy()
    out["baseline_clinical_threshold"] = prf(truth_row, clin)

    resid_tr, scale_tr = compute_residuals(dtrain_i)
    med = dtrain_i[VITALS].median()
    rate = float(make_flags(resid_tr, scale_tr, K).any(axis=1).mean())       # same alert burden as ours
    iso = IsolationForest(n_estimators=100, contamination=min(max(rate, 0.001), 0.5), random_state=seed, n_jobs=-1)
    iso.fit(_features(dtrain_i, resid_tr, med))
    pred_if = iso.predict(_features(dtest_i, resid, med)) == -1
    out["baseline_isolation_forest"] = prf(truth_row, pred_if)
    return out, ep, dtest_i, inj

In [ ]:
"""Figures for the poster / README (matplotlib only)."""
import numpy as np
import matplotlib
import matplotlib.pyplot as plt


COLORS = {"artifact": "#e08a00", "real_event": "#c0392b", "review": "#7f8c8d"}


def plot_case(df, episodes, caseid, path, hours=None):
    """Five vitals of one patient with detected episodes shaded (orange = artifact, red = real event)."""
    d = df[df.caseid == caseid]
    e = episodes[episodes.caseid == caseid]
    fig, axes = plt.subplots(5, 1, figsize=(13, 9), sharex=True)
    for ax, v in zip(axes, VITALS):
        ax.plot(d.time_s / 60, d[v], lw=0.7, color="#2c3e50")
        for _, r in e.iterrows():
            if r["label"] == "review":
                continue
            ax.axvspan(r.start_s / 60 - 0.3, r.end_s / 60 + 0.3, color=COLORS[r["label"]],
                       alpha=0.9 if r["vital"] == v else 0.25, lw=0)
        ax.set_ylabel(f"{v}\n({UNITS[v]})")
    axes[0].set_title(f"Case {caseid}: detected episodes (orange = artifact, red = real event)")
    axes[-1].set_xlabel("Minutes since recording start")
    plt.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def plot_class_mix(episodes, path):
    tab = episodes.groupby(["vital", "label"]).size().unstack(fill_value=0).reindex(VITALS).fillna(0)
    fig, ax = plt.subplots(figsize=(7, 4)); bottom = np.zeros(len(tab))
    for lab in ["artifact", "real_event", "review"]:
        if lab in tab:
            ax.bar(tab.index, tab[lab], bottom=bottom, label=lab.replace("_", " "), color=COLORS[lab]); bottom += tab[lab].to_numpy()
    ax.set_ylabel("Episodes"); ax.set_title("Detected episodes per vital"); ax.legend()
    plt.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def plot_confusion(conf, path):
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    im = ax.imshow(conf.to_numpy(), cmap="Blues")
    ax.set_xticks(range(conf.shape[1]), [c.replace("pred_", "").replace("_", " ") for c in conf.columns])
    ax.set_yticks(range(conf.shape[0]), [i.replace("true_", "").replace("_", " ") for i in conf.index])
    for i in range(conf.shape[0]):
        for j in range(conf.shape[1]):
            ax.text(j, i, int(conf.iat[i, j]), ha="center", va="center", fontsize=12)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Injected truth"); ax.set_title("Artifact vs real-event separation (test patients)")
    plt.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def plot_sensitivity(sens, path):
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    ax.plot(sens.K, sens.detection_recall, marker="o", label="injected episodes detected")
    ax.plot(sens.K, sens.correct_class_rate, marker="s", label="detected AND classified correctly")
    ax.set_xlabel("Threshold K (x patient-specific scale)"); ax.set_ylabel("Fraction"); ax.set_ylim(0, 1.02)
    ax2 = ax.twinx(); ax2.plot(sens.K, sens.background_episodes_per_hour, color="grey", ls="--", marker="^")
    ax2.set_ylabel("Unmatched episodes per hour (dashed)")
    ax.legend(loc="lower left"); ax.set_title("Effect of the threshold K")
    plt.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

## 2. Load and clean the data (same rules as notebook 1)

In [ ]:
raw = load_vitaldb(n_cases=30, seed=42)
df, n_impossible = clean(raw)
print('Rows:', f'{len(raw):,}', '| patients:', df.caseid.nunique(), '| impossible values removed:', f'{n_impossible:,}')

## 3. Run the detector on the real recordings

In [ ]:
K = 4.0
flags, episodes, resid = detect(df, K=K)
hours = len(df) * 2 / 3600
print(f'{len(episodes):,} episodes in {hours:.0f} recorded hours ({len(episodes)/hours:.1f} per hour)')
mix = episodes.label.value_counts()
display(pd.DataFrame({'episodes': mix, 'share %': (100*mix/len(episodes)).round(1)}))
display(pd.crosstab(episodes.vital, episodes.label).reindex(VITALS))

In [ ]:
best = int(df.dropna(subset=['HR','SpO2','MAP']).groupby('caseid').size().idxmax())
plot_case(df, episodes, best, 'case.png')
plot_class_mix(episodes, 'mix.png')
from IPython.display import Image, display
display(Image('case.png')); display(Image('mix.png'))

In [ ]:
# Which departments show the most artifacts / events?
if 'department' in df:
    dept = episodes.caseid.map(df.groupby('caseid').department.first())
    display(episodes.assign(department=dept).groupby(['department','label']).size().unstack(fill_value=0))

## 4. Evaluation with injected ground truth

In [ ]:
ev, _, _, _ = run_evaluation(df, K=K, seed=0)
conf = ev.pop('confusion'); sens = pd.DataFrame(ev.pop('sensitivity'))
import json; print(json.dumps(ev, indent=2, default=str))
display(conf); display(sens)
plot_confusion(conf, 'confusion.png'); plot_sensitivity(sens, 'sens.png')
display(Image('confusion.png')); display(Image('sens.png'))

## 5. Interpretation (fill in after running)

* What share of episodes were artifacts vs real events?
* Which vital produced most artifacts? (the EDA already showed MAP spikes in the arterial line)
* How did the detector compare with the clinical-threshold and Isolation-Forest baselines?
* Limitations: injected (not clinician-labelled) truth; one common artifact source (e.g. electrocautery) can disturb several vitals at once and look like a "real event"; 5-minute baseline is slow to adapt after a genuine level shift.

In [ ]:
# Save everything for the GitHub repo (results/ folder) and download it
import os, zipfile
os.makedirs('results/figures', exist_ok=True)
episodes.to_csv('results/episodes.csv', index=False); conf.to_csv('results/confusion_matrix.csv'); sens.to_csv('results/sensitivity_K.csv', index=False)
for f, t in (('case.png','detected_episodes_case.png'),('mix.png','episode_mix_by_vital.png'),('confusion.png','confusion_matrix.png'),('sens.png','sensitivity_K.png')):
    os.replace(f, 'results/figures/'+t)
metrics = {'cases': int(df.caseid.nunique()), 'rows': int(len(df)), 'hours': round(hours,1), 'impossible_values_removed': int(n_impossible), 'K': K,
           'episodes_total': int(len(episodes)), 'episodes_by_class': {k:int(v) for k,v in mix.items()}, 'evaluation': ev}
json.dump(metrics, open('results/metrics.json','w'), indent=2, default=str)
with zipfile.ZipFile('results.zip','w') as z:
    for root,_,fs in os.walk('results'):
        for f in fs: z.write(os.path.join(root,f))
try:
    from google.colab import files; files.download('results.zip')
except ImportError: pass